In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_model(model, dataloader, class_names, device):
    """
    Evaluates a trained PyTorch model on a test DataLoader.
    
    Returns:
        results = {
            "accuracy": float,
            "classification_report": str,
            "specificity": dict,
            "confusion_matrix": np.ndarray,
            "labels": list,
            "preds": list
        }
    """
    
    model.eval()
    all_preds = []
    all_labels = []

    # Collect predictions and labels
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    # ----- Accuracy -----
    accuracy = accuracy_score(all_labels, all_preds)

    # ----- Classification Report -----
    cls_report = classification_report(all_labels, all_preds, target_names=class_names)

    # ----- Confusion Matrix -----
    cm = confusion_matrix(all_labels, all_preds)

    # ----- Specificity per class -----
    specificities = {}
    for i, class_name in enumerate(class_names):
        TN = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        FP = cm[:, i].sum() - cm[i, i]
        specificity = TN / (TN + FP)
        specificities[class_name] = float(specificity)

    # Pack results
    results = {
        "accuracy": float(accuracy),
        "classification_report": cls_report,
        "specificity": specificities,
        "confusion_matrix": cm,
        "labels": all_labels,
        "preds": all_preds
    }

    return results
